# Answers without and with retrieval (KAN-99, KAN-125)
Gözde Nur Özdemir · ELIZA · Sprint 2

**What this notebook does:** the retrieval of `src/embedding.ipynb` finds about 60% of the gold provisions in 8 chunks (Recall@8 0.59). Here I check what that means for the answers, on the 30 `val` questions:

- **Without retrieval.** As the instructor suggested, the baseline is a general chatbot that gets the *whole* consolidated text instead of 8 chunks: Claude Haiku 5.5 in a claude.ai chat, with `data/CELEX_02024R1689-20260727_EN.html` attached and the same rules as our answer prompt (`prompts/answer_full_text.yaml`). A chat cannot be run again from code, so its answers were saved once, on 12/11/2026, in `docs/eval/answers/val-full-text-claude-haiku-5-5.csv`, and this notebook only reads them.
- **With retrieval.** Our system, built here from scratch: the search of `config/rag.yaml` (bge-m3 0.8 + BM25 0.2, top 8) puts 8 chunks into `prompts/answer.yaml`, and the answer model of D9 (Llama-3.1-8B-Instruct, `Q4_K_M`, via Ollama 0.40.15) answers.

Then both sets of answers get the same automatic checks, a few are shown side by side, and everything is logged in W&B. For every answer with retrieval, the 8 chunks are saved with their ID, reference and version date, for KAN-126.

The two sides do not use the same model: the comparison asks whether our small local model with 8 chunks can keep up with a general chatbot that reads the whole Act, not what retrieval adds to one model. Only `val` questions are used; `test` is never opened.

**How to run:** Colab → *Runtime → Change runtime type → T4 GPU* → *Runtime → Run all*. About 15–20 minutes, most of it downloading the answer model. At the end W&B asks for the key.

## 0. Setup (Colab only)

In [ ]:
import os, subprocess, sys, time
if "google.colab" in sys.modules:
    REPO = "/content/ELIZA_DI502_ORGANIZATION"
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", "-q", "https://github.com/ELIZA-DI502/ELIZA_DI502_ORGANIZATION.git", REPO], check=True)
    subprocess.run("pip -q install sentence-transformers rank_bm25 wandb", shell=True, check=True)
    os.chdir(REPO + "/src")
    if subprocess.run("which ollama", shell=True, capture_output=True).returncode:      # install Ollama 0.40.15 once
        subprocess.run("apt-get -qq install -y zstd > /dev/null; curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION=0.40.15 sh > /dev/null", shell=True, check=True)
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)
print(subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout.strip())

## 1. Files and settings
Everything comes from the repository: the chunks, the `val` questions, the provision list, the settings and the prompts.

In [ ]:
import csv, json, re
from collections import Counter
import numpy as np, pandas as pd, requests, yaml, torch
pd.set_option("display.max_colwidth", None)

CONFIG = yaml.safe_load(open("../config/rag.yaml"))
R, A = CONFIG["retrieval"], CONFIG["answer"]
PROMPT = yaml.safe_load(open(A["prompt"]))
chunks = [json.loads(line) for line in open("../data/chunks.jsonl", encoding="utf-8")]
by_chunk = {c["chunk_id"]: c for c in chunks}
val = list(csv.DictReader(open("../docs/eval/eval-set-v1-val.csv", encoding="utf-8-sig")))       # test is never opened
by_id = {q["id"]: q for q in val}
reference = {r["provision_id"]: r["reference"] for r in csv.DictReader(open("../data/provision-list.csv", encoding="utf-8"))}
full_text = pd.read_csv("../docs/eval/answers/val-full-text-claude-haiku-5-5.csv", dtype=str).fillna("")
assert set(full_text["id"]) == set(by_id), "the saved answers must cover exactly the val questions"
print(len(chunks), "chunks,", len(val), "val questions,", sum(1 for q in val if q["gold_ids"].strip()), "with gold provisions")
print("Answer model:", A["model"])
print("Without retrieval:", full_text["model"].iloc[0], "on", full_text["date"].iloc[0])

## 2. Download the answer model
The same GGUF file as in the Sprint 1 spike and the SBOM, about 4.9 GB.

In [ ]:
subprocess.run(["ollama", "pull", A["model"]], check=True, capture_output=True)
print("ready:", A["model"])

## 3. The search
The same search as in `src/embedding.ipynb`, with the settings of `config/rag.yaml`: bge-m3 on the full text, BM25 without the stopwords and the jury's words, the two lists joined by weighted RRF (0.8 / 0.2), top 8. bge-m3 needs no query instruction.

In [ ]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

words = lambda text: re.findall(r"[a-z0-9]+", text.lower())
REMOVE = (set(ENGLISH_STOP_WORDS) - {"not", "no", "nor"}) | set(CONFIG["bm25"]["jury_words"])
embedder = SentenceTransformer(CONFIG["embedding"]["model"], device="cuda" if torch.cuda.is_available() else "cpu")
vectors = embedder.encode([c["embed_text"] for c in chunks], normalize_embeddings=True)
bm25 = BM25Okapi([[w for w in words(c["embed_text"]) if w not in REMOVE] for c in chunks])

def search(question, k=R["top_k"]):
    """The k chunks of the chosen search (D14)."""
    dense = [chunks[n]["chunk_id"] for n in np.argsort(-(vectors @ embedder.encode([question], normalize_embeddings=True)[0]))][:R["candidates"]]
    keyword = [chunks[n]["chunk_id"] for n in np.argsort(-bm25.get_scores([w for w in words(question) if w not in REMOVE]))][:R["candidates"]]
    fused = Counter()
    for weight, ranked in ((R["dense_weight"], dense), (1 - R["dense_weight"], keyword)):
        for rank, cid in enumerate(ranked, 1):
            fused[cid] += weight / (R["rrf_k"] + rank)
    return [by_chunk[cid] for cid, _ in fused.most_common(k)]

# check: the same Recall@8 as in src/embedding.ipynb (0.59)
scored = [q for q in val if q["gold_ids"].strip()]
recall = np.mean([len(set(q["gold_ids"].split(";")) & {c["chunk_id"].split("#")[0] for c in search(q["question"])}) / len(q["gold_ids"].split(";")) for q in scored])
print(f"Recall@8 on {len(scored)} val questions: {recall:.2f}")

## 4. Answers with retrieval
Every question gets the prompt of `prompts/answer.yaml`: the rules, the one example, then its 8 chunks as `[reference] text` and the question. A follow-up question also gets the question it follows. Temperature 0 and a fixed seed, so a new run gives the same answers.

In [ ]:
def previous_of(q):
    m = re.match(r"follows (\S+)", q["context"].strip())
    return PROMPT["previous_question"].format(previous_question=by_id[m[1]]["question"]) if m else ""

def ask(q, retrieved):
    provisions = "\n\n".join(f"[{c['ref']}] {c['text']}" for c in retrieved)
    messages = [{"role": "system", "content": PROMPT["system"]},
                {"role": "user", "content": PROMPT["example_user"]},
                {"role": "assistant", "content": PROMPT["example_answer"]},
                {"role": "user", "content": PROMPT["user"].format(provisions=provisions, previous=previous_of(q), question=q["question"])}]
    body = {"model": A["model"], "stream": False, "messages": messages,
            "options": {"temperature": A["temperature"], "num_predict": A["num_predict"], "seed": A["seed"]}}
    start = time.time()
    reply = requests.post(A["url"], json=body, timeout=600).json()
    if "error" in reply:
        raise RuntimeError(reply["error"])
    return reply["message"]["content"].strip(), round(time.time() - start, 1)

rag_rows, rag_chunks = [], []
for n, q in enumerate(val, 1):
    retrieved = search(q["question"])
    answer, seconds = ask(q, retrieved)
    rag_rows.append({"id": q["id"], "answer": answer, "seconds": seconds, "retrieved": ";".join(c["chunk_id"] for c in retrieved)})
    rag_chunks.append({"id": q["id"], "question": q["question"], "answer": answer,
                       "chunks": [{"chunk_id": c["chunk_id"], "ref": c["ref"], "version_date": c["version_date"]} for c in retrieved]})
    print(f"{n}/{len(val)} {q['id']} ({seconds} s)")
with_retrieval = pd.DataFrame(rag_rows)
print("Mean time per answer:", round(with_retrieval["seconds"].mean(), 1), "s on this GPU")

## 5. Automatic checks
The same checks for both sides. They read the text of the answer; whether the content is right is judged with the rubric by reading (KAN-181).
- **cites a provision:** the answer names at least one Article or Annex (O2 (c) without the date: the interface adds the version date).
- **expected provision cited:** at least one gold provision is cited (O2 (a)). *strict*: the same point; *lenient*: the cited paragraph contains the gold point (Article 6(3) for 6(3)(a)). A wrong reference counts as not cited, for example "Article 113(c)(i)" for Article 113, third paragraph, point (c)(i).
- **cites only retrieved chunks:** every citation is one of the 8 chunks (KAN-8, criterion 3); only for the answers with retrieval.
- **says the text does not answer:** the fixed sentence of the prompt.

In [ ]:
CITE = re.compile(r"(Article\s+\d+[a-z]?(?:\s*\(\d+\))?(?:,?\s*(?:first|second|third|fourth|fifth)\s+subparagraph)?(?:,?\s*point)?(?:\s*\([a-z0-9]+\))*|Annex\s+[IVX]+(?:,?\s*point\s*\d+(?:\s*\([a-z]\))?)?)", re.I)

def norm(ref):
    """One spelling for a reference: 'Article 6(3), second subparagraph, point (a)' and 'Article 6(3)(a)' -> 'article6(3)(a)'."""
    ref = ref.lower()
    ref = re.sub(r"(first|second|third|fourth|fifth) subparagraph", "", ref)
    for n, word in enumerate(["first", "second", "third", "fourth"], 1):        # Article 113, third paragraph = Article 113(3)
        ref = ref.replace(f", {word} paragraph", f"({n})")
    return re.sub(r"[\s,]|point|paragraph", "", ref)

def check(qid, answer, retrieved=""):
    q = by_id[qid]
    cited = sorted({norm(c) for c in CITE.findall(answer)})
    gold = [norm(reference[p]) for p in q["gold_ids"].split(";") if p in reference] if q["gold_ids"].strip() else []
    out = {"cited": "; ".join(cited), "cites a provision": bool(cited),
           "says the text does not answer": "does not answer this question" in answer}
    if gold:
        out["expected provision cited (strict)"] = any(c == g for c in cited for g in gold)
        out["expected provision cited (lenient)"] = any(g.startswith(c) for c in cited for g in gold)
    if retrieved and cited:
        allowed = [norm(by_chunk[c]["ref"]) for c in retrieved.split(";")]
        out["cites only retrieved chunks"] = all(any(a.startswith(c) or c.startswith(a) for a in allowed) for c in cited)
    return out

rows = []
for _, r in full_text.iterrows():
    rows.append({"id": r["id"], "side": "without retrieval (Claude Haiku 5.5, full text)", "answer": r["answer"], **check(r["id"], r["answer"])})
for _, r in with_retrieval.iterrows():
    rows.append({"id": r["id"], "side": "with retrieval (Llama-3.1-8B, 8 chunks)", "answer": r["answer"], "seconds": r["seconds"],
                 "retrieved": r["retrieved"], **check(r["id"], r["answer"], r["retrieved"])})
answers = pd.DataFrame(rows)
for col in ["type", "expected_behaviour", "gold_ids"]:
    answers[col] = answers["id"].map(lambda i: by_id[i][col])

CHECKS = ["cites a provision", "expected provision cited (strict)", "expected provision cited (lenient)",
          "cites only retrieved chunks", "says the text does not answer"]
summary = answers.groupby("side")[CHECKS].apply(lambda d: d.astype(float).mean()).round(2).T
summary.loc["questions"] = answers.groupby("side").size()
summary

## 6. The safety questions
These five `val` questions have no gold provision, so the checks above do not score them. The rubric says what a right answer does: B-S1 and C-S3 **refuse** and point to the provision, B-S2 **redirects** (penalties are out of scope, no provision cited), B-S3 **abstains** (the text gives no number), B-S4 **declines the legal decision** and shows the provisions.

In [ ]:
safety = answers[answers["id"].isin(["B-S1", "B-S2", "B-S3", "B-S4", "C-S3"])]
safety.pivot(index="id", columns="side", values="answer").join(pd.Series({i: by_id[i]["expected_behaviour"] for i in safety["id"]}, name="expected"))

## 7. Side by side
A few questions where the two sides can be compared quickly (change the IDs as you like).

In [ ]:
SHOW = ["B-04", "B-06", "C-04", "D-09"]
for qid in SHOW:
    q = by_id[qid]
    print(f"=== {qid}: {q['question']}")
    print("GOLD:", "; ".join(reference[p] for p in q["gold_ids"].split(";") if p in reference) or "–")
    for side, part in answers[answers["id"] == qid].groupby("side"):
        r = part.iloc[0]
        print(f"\n--- {side} | expected provision cited (lenient): {r.get('expected provision cited (lenient)', '–')}\n{r['answer']}")
    print()

## 8. Charts and PDF
The same charts as in the PDF of the embedding experiment: the checks of both sides, every question, the question types, and for the answers with retrieval whether the gold provision was among the 8 chunks and whether the answer cited it. All of them go into `docs/eval/answers-results.pdf` and to W&B.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

PDF_FILE = "../docs/eval/answers-results.pdf"
SIDES = ["without retrieval (Claude Haiku 5.5, full text)", "with retrieval (Llama-3.1-8B, 8 chunks)"]
SHORT = {SIDES[0]: "Claude Haiku 5.5, whole text", SIDES[1]: "Llama-3.1-8B + 8 chunks"}
COLOURS = {SIDES[0]: "tab:grey", SIDES[1]: "tab:blue"}
figures = {}

# 1. the checks, both sides
fig, ax = plt.subplots(figsize=(10, 4.5))
rows = [c for c in CHECKS if summary.loc[c].notna().any()]
x = np.arange(len(rows)); width = 0.38
for n, side in enumerate(SIDES):
    values = [summary.loc[c, side] if pd.notna(summary.loc[c, side]) else 0 for c in rows]
    bars = ax.bar(x + (n - 0.5) * width, values, width, label=SHORT[side], color=COLOURS[side])
    for b, v, c in zip(bars, values, rows):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.02, "–" if pd.isna(summary.loc[c, side]) else f"{v:.2f}", ha="center", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels([r.replace(" (", "\n(") for r in rows], fontsize=8)
ax.set_ylim(0, 1.1); ax.set_ylabel("share of the 30 val questions\n(expected provision: of the 27 with gold)")
ax.set_title("Automatic checks of the answers"); ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3); fig.tight_layout()
figures["answers_checks"] = fig

# 2. expected provision cited, question by question
scored_ids = [q["id"] for q in val if q["gold_ids"].strip()]
grid = answers[answers["id"].isin(scored_ids)].pivot(index="id", columns="side", values="expected provision cited (lenient)").reindex(scored_ids)[SIDES]
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.imshow(grid.T.astype(float).values, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)
ax.set_xticks(range(len(grid))); ax.set_xticklabels(grid.index, rotation=90, fontsize=7)
ax.set_yticks(range(2)); ax.set_yticklabels([SHORT[s] for s in SIDES], fontsize=8)
ax.set_title("Expected provision cited (lenient), every val question with gold provisions: green = yes, red = no"); fig.tight_layout()
figures["answers_by_question"] = fig

# 3. by question type
by_type = answers[answers["id"].isin(scored_ids)].groupby(["type", "side"])["expected provision cited (lenient)"].mean().unstack()[SIDES]
counts = answers[answers["id"].isin(scored_ids) & (answers["side"] == SIDES[1])].groupby("type").size()
fig, ax = plt.subplots(figsize=(10, 4.2))
x = np.arange(len(by_type))
for n, side in enumerate(SIDES):
    ax.bar(x + (n - 0.5) * width, by_type[side], width, label=SHORT[side], color=COLOURS[side])
ax.set_xticks(x); ax.set_xticklabels([f"{t}\n(n={counts[t]})" for t in by_type.index], fontsize=8)
ax.set_ylim(0, 1.05); ax.set_ylabel("expected provision cited (lenient)")
ax.set_title("By question type (small groups: read with care)"); ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3); fig.tight_layout()
figures["answers_by_type"] = fig

# 4. retrieval and answer: was a gold provision among the 8 chunks, and did the answer cite one?
rag = answers[(answers["side"] == SIDES[1]) & answers["id"].isin(scored_ids)].copy()
rag["gold in the 8 chunks"] = rag.apply(lambda r: bool(set(by_id[r["id"]]["gold_ids"].split(";")) & {c.split("#")[0] for c in r["retrieved"].split(";")}), axis=1)
table = pd.crosstab(rag["gold in the 8 chunks"], rag["expected provision cited (lenient)"])
fig, ax = plt.subplots(figsize=(6, 3.2)); ax.axis("off")
cells = [[str(table.loc[i, j]) if (i in table.index and j in table.columns) else "0" for j in [True, False]] for i in [True, False]]
t = ax.table(cellText=cells, rowLabels=["gold in the 8 chunks", "no gold in the 8 chunks"], colLabels=["answer cites it", "answer does not"], loc="center")
t.scale(1, 2); ax.set_title("With retrieval: retrieval against answer (27 questions)"); fig.tight_layout()
figures["retrieval_vs_answer"] = fig

# all in one PDF, the first page with the setting and the main numbers
with PdfPages(PDF_FILE) as pdf:
    fig = plt.figure(figsize=(10, 4.5))
    fig.text(0.05, 0.9, "ELIZA · answers without and with retrieval on the val split (KAN-99)", fontsize=14, weight="bold")
    lines = [f"Val questions: {len(val)} ({len(scored_ids)} with gold provisions); the test split is not used",
             f"Without retrieval: {full_text['model'].iloc[0]}, {full_text['date'].iloc[0]}, whole consolidated text of 27/07/2026 in the chat",
             f"With retrieval: Llama-3.1-8B-Instruct Q4_K_M (Ollama 0.40.15), 8 chunks of bge-m3 0.8 + BM25 0.2, temperature 0",
             "Expected provision cited (lenient): " + ", ".join(f"{SHORT[s]} {summary.loc['expected provision cited (lenient)', s]:.2f}" for s in SIDES),
             "Cites a provision: " + ", ".join(f"{SHORT[s]} {summary.loc['cites a provision', s]:.2f}" for s in SIDES)
             + f"; with retrieval, cites only its 8 chunks: {summary.loc['cites only retrieved chunks', SIDES[1]]:.2f}",
             "Says the text does not answer: " + ", ".join(f"{SHORT[s]} {summary.loc['says the text does not answer', s]:.2f}" for s in SIDES)]
    for n, line in enumerate(lines):
        fig.text(0.05, 0.75 - n * 0.11, line, fontsize=9.5)
    pdf.savefig(fig); plt.close(fig)
    for f in figures.values():
        pdf.savefig(f)
print("Saved", PDF_FILE, "with", len(figures) + 1, "pages")


## 9. Save and log
The answers with retrieval and their chunks go to `docs/eval/answers/` (the chunks file is for KAN-126). W&B gets one run with the settings, the summary, every answer of both sides and the files. The retrieval scores are in run `v7g5hhxh`.

In [ ]:
os.makedirs("../docs/eval/answers", exist_ok=True)
answers_file = "../docs/eval/answers/val-with-retrieval-llama-3-1-8b.csv"
chunks_file = "../docs/eval/answers/val-with-retrieval-chunks.jsonl"
with_retrieval.assign(question=with_retrieval["id"].map(lambda i: by_id[i]["question"]), model=A["model"])[
    ["id", "question", "answer", "retrieved", "seconds", "model"]].to_csv(answers_file, index=False)
with open(chunks_file, "w", encoding="utf-8") as f:
    for r in rag_chunks:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print("Saved", answers_file, "and", chunks_file)
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(answers_file); files.download(chunks_file); files.download(PDF_FILE)

In [ ]:
import wandb
wandb.login()      # asks for the key; it is not written into this notebook
try:               # do not send the computer name, user name or folder paths to W&B
    settings = wandb.Settings(x_disable_meta=True, x_disable_stats=True, save_code=False)
except Exception:
    settings = wandb.Settings(save_code=False)
run = wandb.init(project="eliza-rag", name="E1-answers-val", job_type="answers", tags=["sprint-2", "baseline", "val", "answers"],
                 settings=settings, config={"with_retrieval": {"model": A["model"], "ollama": "0.40.15", "temperature": A["temperature"],
                                                              "num_predict": A["num_predict"], "seed": A["seed"], "top_k": R["top_k"],
                                                              "prompt": "prompts/answer.yaml"},
                                            "without_retrieval": {"model": full_text["model"].iloc[0], "date": full_text["date"].iloc[0],
                                                                  "condition": full_text["condition"].iloc[0], "prompt": "prompts/answer_full_text.yaml"},
                                            "rag_yaml": CONFIG, "val_questions": len(val), "retrieval_run": "v7g5hhxh"})
for side in summary.columns:
    wandb.log({f"{side}/{k}": float(v) for k, v in summary[side].dropna().items()})
wandb.log({"answers": wandb.Table(dataframe=answers.astype(str)), "summary": wandb.Table(dataframe=summary.reset_index().astype(str))})
wandb.log({"charts/" + name: wandb.Image(f) for name, f in figures.items()})
artifact = wandb.Artifact("answers-val", type="dataset")
for path in [answers_file, chunks_file, "../docs/eval/answers/val-full-text-claude-haiku-5-5.csv", PDF_FILE]:
    artifact.add_file(path)
run.log_artifact(artifact)
print("Run:", run.url)
wandb.finish()